In [2]:
import numpy as np
from pathlib import Path
import pandas as pd

In [3]:
DATA_DIR = Path("..") / "data"

files = [
    "data_v100_train.csv",
    "data_v100_val.csv",
    "data_v100_test.csv",
    "data_v100_train_ext.csv",
    "data_v100_val_ext.csv",
    "data_v100_test_ext.csv",
    "test.csv",
    "test_ext.csv",
]

dfs = {name.replace(".csv", ""): pd.read_csv(DATA_DIR / name) for name in files}

for name, df in dfs.items():
    print(f"{name:25s} shape={df.shape}  cols={list(df.columns)}")

data_v100_train           shape=(1866, 4)  cols=['id', 'citation_context', 'section', 'citation_intent']
data_v100_val             shape=(330, 4)  cols=['id', 'citation_context', 'section', 'citation_intent']
data_v100_test            shape=(550, 4)  cols=['id', 'citation_context', 'section', 'citation_intent']
data_v100_train_ext       shape=(1866, 4)  cols=['id', 'citation_context', 'section', 'citation_intent']
data_v100_val_ext         shape=(330, 4)  cols=['id', 'citation_context', 'section', 'citation_intent']
data_v100_test_ext        shape=(550, 4)  cols=['id', 'citation_context', 'section', 'citation_intent']
test                      shape=(326, 3)  cols=['id', 'citation_context', 'section']
test_ext                  shape=(326, 3)  cols=['id', 'citation_context', 'section']


In [4]:
train, val, test_pub = dfs["data_v100_train"], dfs["data_v100_val"], dfs["data_v100_test"]

print("Nulls:\n", pd.concat([train.isnull().sum(), val.isnull().sum(), test_pub.isnull().sum()], axis=1,
                             keys=["train", "val", "test_pub"]))

print("\nDuplicate ids within train:", train["id"].duplicated().sum())
print("id overlap train/val:", set(train.id) & set(val.id))
print("id overlap train/test_pub:", set(train.id) & set(test_pub.id))

label_names = {0: "Background", 1: "Basis", 2: "Discuss", 3: "Differ", 4: "Support"}
dist = pd.concat(
    {name: dfs[name]["citation_intent"].value_counts(normalize=True).sort_index()
     for name in ["data_v100_train", "data_v100_val", "data_v100_test"]},
    axis=1,
)
dist.index = dist.index.map(label_names)
print("\nClass distribution (proportion):\n", dist)

Nulls:
                   train  val  test_pub
id                    0    0         0
citation_context      0    0         0
section               0    0         0
citation_intent       0    0         0

Duplicate ids within train: 0
id overlap train/val: set()
id overlap train/test_pub: set()

Class distribution (proportion):
                  data_v100_train  data_v100_val  data_v100_test
citation_intent                                                
Background              0.790461       0.742424        0.763636
Basis                   0.105573       0.124242        0.134545
Discuss                 0.043944       0.066667        0.054545
Differ                  0.019829       0.021212        0.018182
Support                 0.040193       0.045455        0.029091


Severe imbalance, consistent across splits (~79% Background, ~11% Basis, ~2-7% each for Discuss/Differ/Support). A naive "always predict Background" gets ~76-79% accuracy but tanks Macro F1 (~0.18) — so Macro F1 is the real challenge and we'll need class-weighting/resampling/focal loss later, and must never just optimize accuracy.

In [5]:
print("Unique sections (train):", train["section"].nunique(), "out of", len(train))
print(train["section"].value_counts().head(20))

print("\nSection case variety sample:")
print(sorted(train["section"].str.lower().unique())[:20])

print("\nSection value overlap train/val:",
      len(set(train.section.str.lower()) & set(val.section.str.lower())),
      "/", train["section"].str.lower().nunique())

Unique sections (train): 191 out of 1866
section
giriş                                                  703
materyal ve metot                                       67
yapay zekâ temelli akilli rehabilitasyon teknikleri     60
ilgili çalişmalar                                       57
yöntem                                                  53
literatür taramasi                                      43
bulgular                                                33
ilgili çalışmalar                                       32
literatür taraması                                      27
literatür incelemesi                                    26
önceki çalişmalar                                       20
kavramlar                                               20
benzer çalışmalar                                       19
sonuç                                                   19
yöntemler                                               17
literatür araştırması                                   15
değerle

section is essentially free-text (191 unique values, many just spelling/diacritic variants of the same header — e.g. literatür taraması vs literatür taramasi vs literatür araştırması), and only 43% of train sections reappear in val. Raw one-hot section won't generalize — we'll need to bucket it into ~6 canonical categories (Introduction/Related Work/Method/Results/Discussion/Conclusion) via keyword matching in 03_FE.ipynb, not use it as-is.

In [6]:
for name, df in [("train", train), ("val", val), ("test_pub", test_pub)]:
    df["n_chars"] = df["citation_context"].str.len()
    df["n_words"] = df["citation_context"].str.split().str.len()
    df["n_cite"] = df["citation_context"].str.count("<CITE>")
    df["n_ref"] = df["citation_context"].str.count(r"\[REF\]")

print(train[["n_chars", "n_words", "n_cite", "n_ref"]].describe())

print("\nRows with <CITE> count != 1 (train):", (train["n_cite"] != 1).sum())
print(train.loc[train["n_cite"] != 1, ["id", "citation_context", "n_cite"]].head(5).to_string())

print("\nAvg n_words, n_ref by class:")
print(train.groupby("citation_intent")[["n_words", "n_cite", "n_ref"]].mean())

           n_chars      n_words  n_cite        n_ref
count  1866.000000  1866.000000  1866.0  1866.000000
mean    180.952304    22.875670     1.0     0.012326
std      85.565956    10.595238     0.0     0.132448
min      40.000000     5.000000     1.0     0.000000
25%     123.000000    16.000000     1.0     0.000000
50%     165.000000    21.000000     1.0     0.000000
75%     212.750000    27.000000     1.0     0.000000
max     780.000000   102.000000     1.0     2.000000

Rows with <CITE> count != 1 (train): 0
Empty DataFrame
Columns: [id, citation_context, n_cite]
Index: []

Avg n_words, n_ref by class:
                   n_words  n_cite     n_ref
citation_intent                             
0                22.650169     1.0  0.008136
1                19.903553     1.0  0.000000
2                28.621951     1.0  0.000000
3                25.081081     1.0  0.108108
4                27.746667     1.0  0.093333


CITE appears exactly once in every row (reliable anchor — good for position-based features later). REF (co-citations) is rare overall but notably more common in Differ (0.108) and Support (0.093) than other classes — makes sense, since contrasting/supporting a work often involves citing multiple related studies together. Sentence length also differs mildly: Basis sentences are shortest (~20 words), Discuss longest (~29 words).

In [7]:
pd.set_option("display.max_colwidth", 200)
for label in [1, 2, 3, 4]:
    print(f"\n=== {label_names[label]} (n={  (train.citation_intent==label).sum() }) ===")
    print(train.loc[train.citation_intent == label, "citation_context"].sample(5, random_state=42).to_string(index=False))


=== Basis (n=197) ===
Fazla mesainin proje ve proje elemanlarına etkilerinin ortaya çıkartılması için anket soruları oluşturulurken Türkdoğan ve diğerleri tarafından fazla mesainin verimlilik üzerindeki etkilerinin ince...
                          Akciğer kanserinin tespiti için yapılan bu çalışmada, Kaggle platformunda açık erişimli popüler akciğer BT tarama görüntülerinden oluşan bir veri seti kullanılmıştır <CITE> .
                                                                                                                                       Okuyucu, bu dönüşüm için detaylı bilgilere <CITE>’den ulaşabilir.
                                                                                                             Geri kalan eşitlikler Gibbs sampling yaklaşımı temel alınarak eğitime devam edilir <CITE> .
                                             Önerilen ESA modeli ve ImageNet <CITE> veri seti üzerinde önceden eğitilmiş olan ResNeXt mimarisi, aynı veri seti üzerinde uygul

clear patterns emerging: Basis = citation is literally the source of a dataset/method being reused ("veri seti kullanılmıştır", "temel alınarak"). Differ = explicit contrast language ("kıyaslandığında", "ancak... yer almamaktadır", "farklı"). Support = performance/metric language that confirms results ("desteklenen", "başarım oranı elde edilmiştir", "yüksek doğrulukta"). Discuss looks noisiest/most overlapping with Background — worth flagging as a likely confusion pair, and Differ vs Support both use comparison/metric language, so they may also get confused by a model.

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import chi2

clean_text = train["citation_context"].str.replace("<CITE>", " ", regex=False).str.replace(r"\[REF\]", " ", regex=True)

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, lowercase=True)
X = vec.fit_transform(clean_text)
feature_names = np.array(vec.get_feature_names_out())

for label in range(5):
    y_bin = (train["citation_intent"] == label).astype(int)
    scores, _ = chi2(X, y_bin)
    top_idx = np.argsort(scores)[-15:][::-1]
    print(f"\n=== Top chi2 terms for {label_names[label]} ===")
    print(list(feature_names[top_idx]))


=== Top chi2 terms for Background ===
['bu çalışmada', 'veri seti', 'kullanılmıştır', 'seti', 'önerilen', 'nsga', 'hedef', 'çalışmamızda', 'çalışmada kullanılan', 'çalışma ve', 'çalışmanın', 'benzer', 'çalışmadan', 'algoritmanın', 'diğerleri']

=== Top chi2 terms for Basis ===
['kullanılmıştır', 'veri seti', 'bu çalışmada', 'seti', 'çalışmamızda', 'oluşturulurken', 'owasp', 'seti kullanılmıştır', 'inceptionv3', 'adadelta ve', 'seçilmiştir', 'kaggle', 'denklem', 've birleştirme', 'hesaplanmaktadır']

=== Top chi2 terms for Discuss ===
['terimi', '14', 'gün', 'duygu analizi', 'yerel olmayan', 'gürültü düzeyinin', 'tahminlemesi', 'hisse', 'düzenlileştirme', 'uygunluk terimi', 'olmayan ortalamalar', 'kullanılabilirlik', 'olmadığı', 'klasik duygu', 'göstergesi']

=== Top chi2 terms for Differ ===
['ve ise', 'dice katsayısı', 'dice', 'çalışma ve', 'psnr', 'verdiğini', 'bu çalışmadan', 'düşüktür', 'yöntem en', 'mbb', 'sağlık hizmetlerinde', 'hizmetlerinde', 'kullanılmasını', 'diyabet', 'hizm

The chi2 top terms for Discuss/Differ/Support are dominated by topic-specific jargon from a handful of papers (e.g. nsga, moeadstm, piceag, moalo — multi-objective evolutionary algorithm names — all clustering under Support; dice, psnr — image segmentation metrics — under Differ), not genuine discourse markers. With only 37-82 examples spread across wildly different academic domains (medicine, CS, algorithms, finance...), a bag-of-words/TF-IDF model risks learning paper topic instead of citation intent and won't generalize to test.csv's unseen topics. This is a strong argument for a contextual embedding model (Turkish BERT) over classic TF-IDF+LogReg for the rare classes, and for treating the earlier qualitative discourse-marker patterns (kıyaslandığında, desteklenen, kullanılmıştır) as more trustworthy signal than raw chi2 word lists. Also confirms Background/Basis share nearly identical top terms — genuinely the hardest pair to separate.

In [9]:
import re

def bucket_section(s: str) -> str:
    s = s.lower()
    if re.search(r"giri|amaç|kavram", s):
        return "intro"
    if re.search(r"ilgili çal|literatür|önceki|benzer çal|kaynak tara|art alan", s):
        return "related_work"
    if re.search(r"yöntem|materyal|metot|tasarım|method", s):
        return "method"
    if re.search(r"bulgu|sonuçlar ve|deney|analiz|değerlendirme|performans", s):
        return "results"
    if re.search(r"tartış|discussion", s):
        return "discussion"
    if re.search(r"^sonuç$|sonuç ve öneri|kapanış", s):
        return "conclusion"
    return "other"

train["section_bucket"] = train["section"].apply(bucket_section)
print(train["section_bucket"].value_counts())

ct = pd.crosstab(train["section_bucket"], train["citation_intent"], normalize="index")
ct.columns = ct.columns.map(label_names)
print(ct.round(3))

section_bucket
intro           748
other           374
related_work    304
method          259
results         130
discussion       28
conclusion       23
Name: count, dtype: int64
citation_intent  Background  Basis  Discuss  Differ  Support
section_bucket                                              
conclusion            0.304  0.000    0.130   0.043    0.522
discussion            0.357  0.071    0.143   0.179    0.250
intro                 0.975  0.012    0.012   0.001    0.000
method                0.579  0.394    0.023   0.000    0.004
other                 0.730  0.168    0.035   0.008    0.059
related_work          0.898  0.007    0.095   0.000    0.000
results               0.254  0.146    0.138   0.208    0.254


this is a major finding: section_bucket is highly predictive and mirrors the discourse structure of a paper:

Introduction (748 rows) → 97.5% Background — almost pure signal.
Related Work (304) → 89.8% Background — same.
Method (259) → 39.4% Basis (by far its highest concentration) — makes sense, methods/datasets being reused live here.
Results/Discussion/Conclusion (181 combined) → where Discuss/Differ/Support concentrate (e.g. Conclusion is 52% Support, Results is ~21% Differ vs ~0% in Intro/Related Work).
This means section (once bucketed) is arguably one of our strongest features — a simple heuristic ("if section=Intro/Related Work, predict Background") would already beat a text-only baseline on those buckets. other is still 374 rows (20%) — worth checking what's falling through the regex.

In [11]:
train.loc[train["section_bucket"] == "other", "section"].value_counts().head(20)

section
yapay zekâ temelli akilli rehabilitasyon teknikleri           60
afet risk yönetiminde yapay zekâ kullanimi                    11
yaygın kullanılan veri kümeleri                               10
uygulama                                                      10
esa katmanları ve sınıflandırıcılar                            9
rehabilitasyon hizmetleri ve yapay zekâ                        9
okunabilirlik indeksleri                                       8
bellek tabanlı makine öğrenme modelleri                        8
ilişkili çalışmalar                                            8
bilgi erişim merkezleri ve uygulamalar                         8
tartişma                                                       7
sonuç ve tartişma                                              7
karinca kolonisi optimizasyonu                                 7
önerilen tanıma sistemi                                        7
sistem için gereken bileşenler                                 6
önerilen metodolo

Most other entries are legitimately paper-specific subsection titles (e.g. yapay zekâ temelli akilli rehabilitasyon teknikleri is one paper's literature-review subsection, karinca kolonisi optimizasyonu is a topic subsection) — some can still be rescued with more keywords (tartişma/ilişkili çalışmalar typos), but a chunk is irreducibly free text. Diminishing returns here; I'll refine the regex slightly in 03_FE.ipynb but won't chase 100% coverage.

One last sanity check before we wrap EDA: does the unlabeled test.csv (the actual leaderboard set) look like the same distribution as train (section buckets, text length) — no distribution shift that would break our assumptions?

In [12]:
test_pub_unlabeled = dfs["test"]
for name, df in [("train", train), ("test.csv (unlabeled)", test_pub_unlabeled)]:
    d = df.copy()
    d["section_bucket"] = d["section"].apply(bucket_section)
    d["n_words"] = d["citation_context"].str.split().str.len()
    print(f"\n--- {name} ---")
    print(d["section_bucket"].value_counts(normalize=True).round(3))
    print("n_words mean/median:", d["n_words"].mean().round(1), d["n_words"].median())


--- train ---
section_bucket
intro           0.401
other           0.200
related_work    0.163
method          0.139
results         0.070
discussion      0.015
conclusion      0.012
Name: proportion, dtype: float64
n_words mean/median: 22.9 21.0

--- test.csv (unlabeled) ---
section_bucket
other           0.337
intro           0.233
method          0.227
results         0.107
related_work    0.083
discussion      0.006
conclusion      0.006
Name: proportion, dtype: float64
n_words mean/median: 21.6 20.0


Interpretation — flags a real risk: n_words matches well (test: 21.6 vs train: 22.9 — no length shift), but section_bucket distribution is meaningfully different: test.csv has far fewer intro/related_work (23.3%+8.3%=31.6% vs train's 40.1%+16.3%=56.4%) and far more other/method. This could mean either (a) test papers use different subsection-naming conventions our regex doesn't catch as well, and/or (b) the true class balance on the hidden test set may skew somewhat less Background-heavy than train/val. Takeaway: section is a strong feature but we shouldn't over-rely on it — the text-based model needs to carry real weight, and we should validate the final pipeline on data_v100_test (which does have labels) rather than trusting train/val metrics alone.

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score

def eval_variant(train_df, val_df, text_col="citation_context"):
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000)
    Xtr = vec.fit_transform(train_df[text_col].str.replace("<CITE>", " ", regex=False))
    Xval = vec.transform(val_df[text_col].str.replace("<CITE>", " ", regex=False))
    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(Xtr, train_df["citation_intent"])
    preds = clf.predict(Xval)
    return accuracy_score(val_df["citation_intent"], preds), f1_score(val_df["citation_intent"], preds, average="macro")

acc_base, f1_base = eval_variant(train, val)
acc_ext, f1_ext = eval_variant(dfs["data_v100_train_ext"], dfs["data_v100_val_ext"])

print(f"Flat (base):     Accuracy={acc_base:.3f}  Macro F1={f1_base:.3f}")
print(f"Context (ext):   Accuracy={acc_ext:.3f}  Macro F1={f1_ext:.3f}")

Flat (base):     Accuracy=0.806  Macro F1=0.550
Context (ext):   Accuracy=0.800  Macro F1=0.548


 Flat is marginally better on both metrics (Acc 0.806 vs 0.800, Macro F1 0.550 vs 0.548). The gap is small on this single split (330 val rows), but it's never worse, and it comes with less overfitting risk and a simpler pipeline — so this settles the decision: build on data_v100_* (Flat), not _ext, matching the paper's finding. Also useful: TF-IDF + LogisticRegression baseline = 0.55 Macro F1 — this becomes our benchmark number that a Turkish BERT model in 04_training.ipynb needs to beat

In [14]:
from sklearn.metrics import classification_report

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000)
Xtr = vec.fit_transform(train["citation_context"].str.replace("<CITE>", " ", regex=False))
Xval = vec.transform(val["citation_context"].str.replace("<CITE>", " ", regex=False))
clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(Xtr, train["citation_intent"])
preds = clf.predict(Xval)

print(classification_report(val["citation_intent"], preds, target_names=list(label_names.values()), digits=3))

              precision    recall  f1-score   support

  Background      0.867     0.902     0.884       245
       Basis      0.604     0.707     0.652        41
     Discuss      0.583     0.318     0.412        22
      Differ      1.000     0.143     0.250         7
     Support      0.571     0.533     0.552        15

    accuracy                          0.806       330
   macro avg      0.725     0.521     0.550       330
weighted avg      0.805     0.806     0.795       330



Macro F1 (0.550) is dragged down almost entirely by Differ (F1 0.250, recall only 14.3% — model barely ever predicts it, but is 100% precise when it does) and Discuss (F1 0.412, recall 31.8%). Notice macro recall (0.521) << macro precision (0.725) — the model's core failure mode is under-predicting rare classes, not making wrong ones. With only 37 Differ / 82 Discuss training examples spread over wildly different topics, sparse TF-IDF can't generalize their patterns — this is exactly the scenario where a pretrained Turkish BERT's semantic priors (recognizing contrast/support discourse markers even with few fine-tuning examples) should outperform bag-of-words, and where class-imbalance techniques (oversampling Differ/Discuss, focal loss, or synthetic augmentation as the paper suggests) matter most.

In [15]:
from scipy.sparse import hstack, csr_matrix

def add_section_feature(df):
    buckets = pd.get_dummies(df["section"].apply(bucket_section))
    return buckets

def eval_with_section(train_df, val_df, use_section):
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000)
    Xtr_text = vec.fit_transform(train_df["citation_context"].str.replace("<CITE>", " ", regex=False))
    Xval_text = vec.transform(val_df["citation_context"].str.replace("<CITE>", " ", regex=False))

    if use_section:
        tr_sec = add_section_feature(train_df)
        val_sec = add_section_feature(val_df).reindex(columns=tr_sec.columns, fill_value=0)
        Xtr = hstack([Xtr_text, csr_matrix(tr_sec.values)])
        Xval = hstack([Xval_text, csr_matrix(val_sec.values)])
    else:
        Xtr, Xval = Xtr_text, Xval_text

    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(Xtr, train_df["citation_intent"])
    preds = clf.predict(Xval)
    return f1_score(val_df["citation_intent"], preds, average="macro")

for use_sec, label in [(False, "no section"), (True, "with section")]:
    f1_val = eval_with_section(train, val, use_sec)
    f1_test_pub = eval_with_section(train, test_pub, use_sec)
    print(f"{label:15s} -> Macro F1 val={f1_val:.3f}  Macro F1 test_pub(held-out)={f1_test_pub:.3f}")

no section      -> Macro F1 val=0.550  Macro F1 test_pub(held-out)=0.629
with section    -> Macro F1 val=0.540  Macro F1 test_pub(held-out)=0.620


Adding section hurts Macro F1 on both val (0.550→0.540) and the held-out test_pub (0.629→0.620). This isn't just "helps in-sample but fails to generalize" — it's actively counterproductive everywhere. Combined with the literature evidence and the section-distribution shift we found earlier, this is decisive: drop section as a direct model feature. (It can still be interesting for error analysis/reporting, just not as model input.)

Bonus finding: test_pub (550 rows) gives a higher and more stable Macro F1 (0.629) than val (330 rows, 0.550) — val is noisier due to tiny per-class counts (Differ n=7). This means test_pub (data_v100_test) is the more trustworthy metric for model selection, and our real baseline to beat is 0.629 Macro F1, not 0.550.

## EDA Summary

**Data:** 2746 labeled rows total (`train`=1866, `val`=330, `test_pub`=550, i.e. `data_v100_test`), clean (no nulls/duplicate ids/split leakage). 326 unlabeled rows in `test.csv`/`test_ext.csv` are the true leaderboard set. `data_v100_test` has real labels despite being called "test" — treat it as a second, more reliable held-out eval set (larger, less noisy than `val` for rare classes).

**Class imbalance:** ~79% Background, 11% Basis, 4-7% Discuss, 2% Differ, 3-4.5% Support, consistent across splits. This dominates the whole problem — Macro F1 (not accuracy) must drive every modeling decision.

**Citation structure:** `<CITE>` (the target citation) appears exactly once per row. `[REF]` (co-citations) is rare overall but concentrates in Differ/Support rows — comparisons and results-agreement often co-occur with other citations.

**Input choice — Flat vs Ext (verified empirically):** `_ext` (surrounding-sentence context) does not outperform the single-sentence (`Flat`) input on our data (Macro F1 0.548 vs 0.550, TF-IDF+LogReg). Decision: **build features on `data_v100_*` (Flat), not `_ext`.**

**`section` feature (verified empirically):** Bucketing the 191 raw free-text section labels into ~6 canonical categories shows strong in-sample correlation with class (Intro/Related Work ~90-98% Background; Method concentrates Basis; Results/Discussion/Conclusion concentrate Discuss/Differ/Support). However: (a) train vs `test.csv` section-bucket distributions differ meaningfully, and (b) adding `section_bucket` as a model feature *reduced* Macro F1 on both `val` (0.550→0.540) and `test_pub` (0.629→0.620). Decision: **drop `section` as a direct model feature.**

**External validation:** This dataset matches the published Turkish Web-of-Science-taxonomy CIC dataset ([Karaca & Eravcı, UBMK 2025](https://arxiv.org/abs/2509.21907); [companion thesis](https://doi.org/10.48623/aperta.286647)), whose experiments independently confirm both findings above: "Flat" single-sentence classification beat context/section-enriched variants due to spurious correlations on structural metadata, and Background-class dominance was the central obstacle for minority-class (Basis, Differ) recognition.

**Baseline:** TF-IDF (1-2 grams) + `LogisticRegression(class_weight="balanced")` = **0.629 Macro F1** on `data_v100_test`. Weakest classes: **Differ** (F1 0.250, recall 14%) and **Discuss** (F1 0.412, recall 32%) — the main targets for imbalance handling (class weighting, oversampling, or synthetic augmentation) and for a semantically richer model (Turkish BERT) in later notebooks.

**Next:** `03_FE.ipynb` — Flat text cleaning (`<CITE>`/`[REF]` handling), no `section` feature (or only as a weak auxiliary experiment), imbalance-aware feature/label pipeline, then `04_training.ipynb` — Turkish encoder (BERTurk/Turkish ELECTRA/DeBERTaV3) fine-tuning benchmarked against the 0.629 Macro F1 baseline, model-selected on `data_v100_test` rather than `val`.